In [7]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

BASE_DIR = Path.cwd().parent

CHROMA_DIR = BASE_DIR / "chroma_db"

In [8]:
print("Loading embedding model...\n")
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded.\n")

Loading embedding model...



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7925.54it/s]


Embedding model loaded.



In [9]:
print("CHROMA_DIR:", CHROMA_DIR.resolve())
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

print(client.list_collections())

CHROMA_DIR: C:\Users\ADMIN\Desktop\Projects\MLReserachRAG\backend\chroma_db
[Collection(name=research_papers)]


In [10]:
collection = client.get_collection(
    name="research_papers"
)
print("Connected to ChromaDB.\n")

Connected to ChromaDB.



In [30]:
query = "How do modern architectures improve transformer efficiency?"
print(f"Query: {query}\n")

query_embedding = embedding_model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

print("TOP RETRIEVED CHUNKS")

retrieved_docs = results["documents"][0]
retrieved_metadata = results["metadatas"][0]

for i in range(len(retrieved_docs)):

    print(f"\nRESULT {i+1}")
    print("-" * 50)

    print("PAPER:")
    print(retrieved_metadata[i]["paper_title"])

    print("\nCHUNK:")
    print(retrieved_docs[i][:1000])

    print("\n")


Query: How do modern architectures improve transformer efficiency?

TOP RETRIEVED CHUNKS

RESULT 1
--------------------------------------------------
PAPER:
Improving language understanding by generative pre-training

CHUNK:
. Finally, we also compare with our transformer architecture directly trained on supervised target tasks, without pre-training. We observe that the lack of pre-training hurts performance across all the tasks, resulting in a 14.8% decrease compared to our full model.



RESULT 2
--------------------------------------------------
PAPER:
Retentive Network

CHUNK:
Figure 6: Inference cost of Transformer and RetNet with a model size of 6.7B. RetNet outperforms Transformers in terms of memory consumption, throughput, and latency. 

requiring much less GPU memory to host RetNet. The additional memory consumption of RetNet is almost negligible (i.e., about 3%) while the model weights occupy 97%. 

**Throughput** As presented in Figure 6b, the throughput of Transformer drop

In [ ]:
#reranking

import cohere
from dotenv import load_dotenv
import os

In [31]:
load_dotenv()
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
co = cohere.ClientV2(api_key=COHERE_API_KEY)

response = co.rerank(
    model="rerank-v3.5",
    query="How do modern architectures improve transformer efficiency?",
    documents=results["documents"][0],
    top_n=3,
)

print(response)

id='8d7b41f7-d540-489c-9e6c-7e756d8951c7' results=[V2RerankResponseResultsItem(index=4, relevance_score=0.26312268), V2RerankResponseResultsItem(index=0, relevance_score=0.25541142), V2RerankResponseResultsItem(index=3, relevance_score=0.18216383)] meta=ApiMeta(api_version=ApiMetaApiVersion(version='2', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, image_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, cached_tokens=None, warnings=None)
